# Merge local WFDB into `data/evaluation/processed`

Converts leftovers under `data/` (outside `data/evaluation`) so you do not re-download them.

| Dataset | What this notebook does |
|---|---|
| LUDB | Convert **all** records found in `data/LU_DB` |
| QT Database | Convert **all** records found in `data/QT_DB` (native 2-lead) |
| PTB-XL | Convert whatever is already on disk (`evaluation/raw` + stray `data/ptb-xl`) |
| STAFF III | **Do not** convert the full dump. Delete WFDB copies of records that are already processed, so `data/staff_III` shrinks. New STAFF records go through `Download_STAFF_III.ipynb`. |

After a successful convert the source WFDB for that record is deleted. Folder-level cleanup of `data/LU_DB` / `data/QT_DB` is a later step once you inspect the report.

In [ ]:
from pathlib import Path
import sys

REPO = Path.cwd() if (Path.cwd() / "evaluation" / "common.py").exists() else Path.cwd().parent
sys.path.insert(0, str(REPO / "evaluation"))

import pandas as pd
from IPython.display import display

import common as C

OVERWRITE_PROCESSED = False
CONSUME_WFDB = True

catalogs = C.ensure_catalogs()
print("catalogs:")
for key, path in catalogs.items():
    print(f"  {key}: {path}  exists={path.is_file()}")


## LUDB — convert every local record

In [ ]:
ludb_ids = C.list_cache_record_ids("ludb")
# also pick up evaluation/raw leftovers
raw_ludb = C.RAW_ROOT / "ludb" / "data"
if raw_ludb.is_dir():
    ludb_ids = sorted(set(ludb_ids) | {p.stem for p in raw_ludb.glob("*.hea")})
print(f"LUDB local WFDB: {len(ludb_ids)}")
ludb_results = [
    C.acquire_and_convert_ludb(rid, overwrite=OVERWRITE_PROCESSED, consume=CONSUME_WFDB)
    for rid in ludb_ids
]
print("LUDB", C.summarize_acquire(ludb_results))


## QT Database — convert every local record

In [ ]:
qtdb_ids = C.list_cache_record_ids("qtdb")
raw_qt = C.RAW_ROOT / "qtdb"
if raw_qt.is_dir():
    qtdb_ids = sorted(set(qtdb_ids) | {p.stem for p in raw_qt.glob("*.hea")})
print(f"QTDB local WFDB: {len(qtdb_ids)}")
qtdb_results = [
    C.acquire_and_convert_qtdb(rid, overwrite=OVERWRITE_PROCESSED, consume=CONSUME_WFDB)
    for rid in qtdb_ids
]
print("QTDB", C.summarize_acquire(qtdb_results))


## PTB-XL — convert only what is already on disk

In [ ]:
meta, statements = C.load_ptbxl_catalogue()
meta = meta.copy()
meta["record_id"] = meta["ecg_id"].map(lambda x: f"{int(x):05d}_hr")
local_ptb = set(C.list_cache_record_ids("ptb_xl"))
print(f"PTB-XL local WFDB stems: {len(local_ptb)}")
ptb_results = []
for rec_id in sorted(local_ptb):
    rows = meta[meta["record_id"] == rec_id]
    if rows.empty:
        print(f"skip {rec_id}: not in ptbxl_database.csv")
        continue
    row = rows.iloc[0]
    ptb_results.append(
        C.acquire_and_convert_ptbxl(
            rec_id,
            row,
            statements,
            filename_hr=str(row["filename_hr"]),
            overwrite=OVERWRITE_PROCESSED,
            consume=CONSUME_WFDB,
        )
    )
print("PTB-XL", C.summarize_acquire(ptb_results))


## STAFF III — consume cache copies of already-processed records only

In [ ]:
ann = C.load_staff_catalogue()
staff_processed = C.processed_record_ids(C.PROCESSED_ROOT / "staff_iii")
staff_cache = set(C.list_cache_record_ids("staff_iii"))
to_consume = [rid for rid in staff_processed if rid in staff_cache]
print(f"STAFF processed={len(staff_processed)} cache={len(staff_cache)} consume_now={to_consume}")
staff_results = [
    C.acquire_and_convert_staff(rid, ann, overwrite=False, consume=CONSUME_WFDB)
    for rid in to_consume
]
print("STAFF", C.summarize_acquire(staff_results))
print("STAFF cache remaining:", len(C.list_cache_record_ids("staff_iii")))


## Report

In [ ]:
inv = C.dataset_inventory()
display(inv)
C.plot_download_fractions(inv)
print("Cache leftover counts:", {slug: len(C.list_cache_record_ids(slug)) for slug in C.LOCAL_CACHES})
